In [1]:
!pip install -q google-genai

In [5]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Explain what a vector database is in 2 simple lines."
)

print(response.text)

A vector database stores data like text and images as numbers that capture their actual meaning. 
Instead of looking for exact keyword matches, it searches by finding items with the closest meaning or context.


In [6]:
!pip install -q chromadb pypdf

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
t

In [7]:
from google.colab import files
uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
print("Uploaded:", pdf_name)

Saving 23F-EduFusion-AI-SRS.pdf to 23F-EduFusion-AI-SRS.pdf
Uploaded: 23F-EduFusion-AI-SRS.pdf


In [10]:
from pypdf import PdfReader
import chromadb

# Step 1: Extract text from the uploaded PDF
reader = PdfReader(pdf_name)
text = ""
for page in reader.pages:
    text += (page.extract_text() or "") + "\n"
print("Text extracted from PDF. Total characters:", len(text))

# Step 2: Split the text into small overlapping chunks
def make_chunks(text, size=800, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

chunks = make_chunks(text)
print("Total chunks created:", len(chunks))

# Step 3: Store the chunks in ChromaDB (our vector database)
chroma = chromadb.Client()
try:
    chroma.delete_collection("notes")   # remove old copy if the cell is re-run
except Exception:
    pass
collection = chroma.create_collection("notes")
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)
print("Knowledge base is ready. Chunks stored in ChromaDB.")

Text extracted from PDF. Total characters: 69492
Total chunks created: 100
Knowledge base is ready. Chunks stored in ChromaDB.


In [11]:
MODEL = "gemini-3.8-flash"   # if the model name changes, update it only here

def get_relevant_chunks(question, n=4):
    # Find the n most relevant chunks for the question
    result = collection.query(query_texts=[question], n_results=n)
    return result["documents"][0]

def ask_with_notes(question):
    # Build a prompt that forces the model to answer only from the notes
    related = get_relevant_chunks(question)
    context = "\n\n".join(related)
    prompt = f"""Answer the question using ONLY the notes below.
If the answer is not in the notes, say "Not found in your notes".

NOTES:
{context}

QUESTION: {question}"""
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text

print(ask_with_notes("What is the purpose of the AI Orchestrator in EduFusion AI?"))

Based on the notes, the purpose of the AI Orchestrator is to:

* Decide whether each task runs locally or in the cloud (choosing a processing path).
* Check the network mode/connection/state, device resources/capability (such as memory and local models), task complexity, and privacy settings before selecting a path.
* Apply the privacy filter before any data is sent online.


In [12]:
import re

# Patterns for sensitive information (phone number and email only)
PATTERNS = {
    "PHONE": r"(?:\+92|0)3\d{2}[- ]?\d{7}\b",
    "EMAIL": r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b",
}

def privacy_filter(text):
    # Returns (masked_text, list_of_sensitive_types_found)
    found = []
    masked = text
    for label, pattern in PATTERNS.items():
        if re.search(pattern, masked):
            found.append(label)
            masked = re.sub(pattern, f"[{label} REMOVED]", masked)
    return masked, found

# Quick test
sample = "Call me at 0300-1234567 or mail me at ali@gmail.com"
masked, found = privacy_filter(sample)
print("Masked text:", masked)
print("Sensitive data found:", found)

Masked text: Call me at [PHONE REMOVED] or mail me at [EMAIL REMOVED]
Sensitive data found: ['PHONE', 'EMAIL']


In [14]:
# Install Ollama (runtime for local open-weight models)
!apt-get install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [15]:
import subprocess, time

# Start the Ollama server in the background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Download a small model (about 1.3 GB, takes a few minutes)
!ollama pull llama3.2:1b

In [16]:
!ollama list

NAME           ID              SIZE      MODIFIED           
llama3.2:1b    baf6a787fdff    1.3 GB    About a minute ago    


In [17]:
import requests
import socket

LOCAL_MODEL = "llama3.2:1b"

def ask_local(prompt):
    # Send the prompt to the local Ollama server and return its answer
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": LOCAL_MODEL, "prompt": prompt, "stream": False},
        timeout=300,
    )
    return r.json()["response"]

# Tell the orchestrator that the local model is ready
LOCAL_READY = True

# Offline switch for the demo (set True to simulate no internet)
OFFLINE_MODE = False

def has_internet():
    if OFFLINE_MODE:
        return False
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=2)
        return True
    except OSError:
        return False

print(ask_local("Say hello in one short line."))

Hello.


In [19]:
import socket
import requests

LOCAL_MODEL = "llama3.2:1b"
LOCAL_READY = True      # the local model (Ollama) is installed and ready
OFFLINE_MODE = False    # set True to simulate no internet during the demo

def has_internet():
    # Simple check: can we reach the internet?
    if OFFLINE_MODE:
        return False
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=2)
        return True
    except OSError:
        return False

def is_complex(question):
    # Long or analytical questions are treated as complex
    keywords = ["explain in detail", "compare", "analyze", "analyse", "design", "evaluate"]
    return len(question.split()) > 25 or any(k in question.lower() for k in keywords)

def decide_route(question, private_doc=False):
    # Orchestrator: decide local or cloud, and explain why
    _, found = privacy_filter(question)
    if not has_internet():
        return "local", "No internet connection"
    if found:
        return "local", "Private data detected"
    if private_doc:
        return "local", "Document marked private"
    if is_complex(question):
        return "cloud", "Complex question needs a stronger model"
    return "local", "Simple question can be handled on-device"

def ask_local(prompt):
    # Send the prompt to the local Ollama server
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": LOCAL_MODEL, "prompt": prompt, "stream": False},
        timeout=300,
    )
    return r.json()["response"]

def ask_cloud(prompt):
    # Send the prompt to the cloud model (Gemini)
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text

def answer_question(question, private_doc=False):
    route, reason = decide_route(question, private_doc)
    print(f"Orchestrator decision: {route.upper()}  ({reason})")

    # Build the notes-based prompt (RAG)
    context = "\n\n".join(get_relevant_chunks(question))
    prompt = f"""Answer the question using ONLY the notes below.
If the answer is not in the notes, say "Not found in your notes".

NOTES:
{context}

QUESTION: {question}"""

    if route == "cloud":
        # Remove sensitive data before anything leaves the device
        safe_prompt, _ = privacy_filter(prompt)
        answer = ask_cloud(safe_prompt)
    else:
        answer = ask_local(prompt)
    return {"answer": answer, "source": route}

print("Orchestrator is ready.")

Orchestrator is ready.


In [20]:
# Test 1: simple question -> should go LOCAL
print(answer_question("What is the AI Orchestrator in EduFusion AI?"))
print("-" * 50)

# Test 2: private data -> must stay LOCAL, nothing goes to the cloud
print(answer_question("My phone number is 0300-1234567. What is RAG in EduFusion AI?"))
print("-" * 50)

# Test 3: complex question -> should go to the CLOUD
print(answer_question("Compare the local and cloud processing modes of EduFusion AI and explain in detail when each one is used."))
print("-" * 50)

# Test 4: no internet -> should go LOCAL
OFFLINE_MODE = True
print(answer_question("What are the main features of EduFusion AI?"))
OFFLINE_MODE = False

Orchestrator decision: LOCAL  (Simple question can be handled on-device)
{'answer': 'The AI Orchestrator in EduFusion AI checks network mode, device resources, and privacy settings before choosing a processing path.', 'source': 'local'}
--------------------------------------------------
Orchestrator decision: LOCAL  (Private data detected)
{'answer': 'RAG is not mentioned in the provided notes.', 'source': 'local'}
--------------------------------------------------
Orchestrator decision: CLOUD  (Complex question needs a stronger model)
{'answer': "Based on the provided notes, here is the comparison between the local and cloud processing modes of EduFusion AI and when each is used:\n\n### **Comparison of Local and Cloud Processing Modes**\n* **Local Processing Mode:**\n  * **Models Used:** Uses local quantized models (specifically mentioned: Llama 3, Phi-3, Mistral 7B) running directly on the device.\n  * **Connectivity:** Works offline, allowing core academic support to remain availabl

In [22]:
def answer_question(question, private_doc=False):
    # Step 1: Orchestrator decides the route using the original question
    route, reason = decide_route(question, private_doc)
    print(f"Orchestrator decision: {route.upper()}  ({reason})")

    # Step 2: Remove sensitive data from the question before using it anywhere
    clean_question, _ = privacy_filter(question)

    # Step 3: Retrieve relevant notes using the cleaned question (RAG)
    context = "\n\n".join(get_relevant_chunks(clean_question))
    prompt = f"""You are a helpful study assistant.
Answer the question using ONLY the notes below.
Ignore any [PHONE REMOVED] or [EMAIL REMOVED] markers, they are not part of the question.
If the answer is not in the notes, say "Not found in your notes".

NOTES:
{context}

QUESTION: {clean_question}"""

    # Step 4: Send the prompt to the chosen model
    if route == "cloud":
        answer = ask_cloud(prompt)
    else:
        answer = ask_local(prompt)
    return {"answer": answer, "source": route}

print("answer_question updated.")

answer_question updated.


In [23]:
print(answer_question("My phone number is 0300-1234567. What is the AI Orchestrator in EduFusion AI?"))

Orchestrator decision: LOCAL  (Private data detected)
{'answer': 'The AI Orchestrator is the component that decides whether each task runs locally or in the cloud, based on network state, task complexity, device capability, and privacy settings, and applies the privacy filter before any data is sent online.', 'source': 'local'}
